# Notebook 2 — T4 GPU Training (v4 — Anti-Overfit)**Input Datasets (add these in Kaggle sidebar):**1. `assualttger/sdp-preprocessing` — Preprocessed .npy face crops (FF++ + 140K GAN + Celeb-DF)2. `assualttger/final-1` — manifest.json + model_config.yaml from NB1**Anti-overfit fixes vs v3:**- Early stopping (patience=5)- Stronger data augmentation (ColorJitter 0.3, RandomErasing, Grayscale, Affine)- Label smoothing (0.05)- Dropout(0.3) before FC layers in both branches- Layer freezing during warmup epochs- Proper LinearLR warmup → CosineAnnealing schedule

In [ ]:
# == CELL 0: Copy NB1 Files + Write Fixed model.py ==import os, shutildef find_and_copy(filename):    for root, _, files in os.walk("/kaggle/input"):        if filename in files:            src = os.path.join(root, filename)            shutil.copy(src, f"/kaggle/working/{filename}")            print(f"  OK {filename}  <-  {src}")            return True    print(f"  MISSING {filename}")    return Falseprint("Copying NB1 output files:")for f in ["manifest.json", "model_config.yaml"]:    find_and_copy(f)model_py = '''import torch, torch.nn as nn, timm, yamlfrom safetensors.torch import save_file, load_fileclass ConvNextBranch(nn.Module):    def __init__(self, model_name="convnextv2_base", pretrained=True):        super().__init__()        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, drop_path_rate=0.2)        self.backbone.set_grad_checkpointing(enable=True)        self.dropout = nn.Dropout(0.3)        self.fc = nn.Linear(self.backbone.num_features, 1)    def forward(self, x):        features = self.backbone(x)        return self.fc(self.dropout(features)), featuresclass SwinBranch(nn.Module):    def __init__(self, model_name="swinv2_base_window12to16_192to256", pretrained=True):        super().__init__()        try:            self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, drop_path_rate=0.2)        except Exception:            self.backbone = timm.create_model("swinv2_base_window12_256", pretrained=pretrained, num_classes=0, drop_path_rate=0.2)        self.backbone.set_grad_checkpointing(enable=True)        self.dropout = nn.Dropout(0.3)        self.fc = nn.Linear(self.backbone.num_features, 1)    def forward(self, x):        features = self.backbone(x)        return self.fc(self.dropout(features)), featuresclass EnsembleDeepfakeDetector(nn.Module):    def __init__(self, config_path="/kaggle/working/model_config.yaml"):        super().__init__()        cfg = yaml.safe_load(open(config_path))["model"]        self.cnn_branch = ConvNextBranch(cfg["ensemble"]["cnn_branch"], cfg["pretrained"])        self.vit_branch = SwinBranch(cfg["ensemble"]["vit_branch"], cfg["pretrained"])        self.temperature_cnn = nn.Parameter(torch.ones(1))        self.temperature_vit = nn.Parameter(torch.ones(1))    def forward(self, x):        cnn_logits, _ = self.cnn_branch(x)        vit_logits, _ = self.vit_branch(x)        ensemble = (cnn_logits / self.temperature_cnn + vit_logits / self.temperature_vit) / 2.0        return ensemble, cnn_logits / self.temperature_cnn, vit_logits / self.temperature_vit    def save_safetensors(self, path):        save_file(dict(self.state_dict()), path)        print(f"Saved -> {path}")    def load_safetensors(self, path):        self.load_state_dict(load_file(path))        print(f"Loaded <- {path}")'''with open("/kaggle/working/model.py", "w") as f:    f.write(model_py.strip())print("model.py written (with Dropout 0.3 + DropPath 0.2)")

In [ ]:
# == CELL 1: Imports ==import torch, torch.nn as nn, torch.optim as optimfrom torch.utils.data import DataLoader, Datasetfrom torch.cuda.amp import GradScaler, autocastfrom torchvision import transformsfrom PIL import Imageimport numpy as np, yaml, json, os, sys, math, timeimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import (    matthews_corrcoef, brier_score_loss, roc_auc_score,    confusion_matrix, precision_score, recall_score, f1_score)from tqdm.notebook import tqdmimport warnings; warnings.filterwarnings("ignore")sys.path.insert(0, "/kaggle/working")device = torch.device("cuda" if torch.cuda.is_available() else "cpu")print(f"Device : {device}")if torch.cuda.is_available():    print(f"GPU    : {torch.cuda.get_device_name(0)}")    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# == CELL 2: Hyperparameters (Anti-Overfit) ==config   = yaml.safe_load(open("/kaggle/working/model_config.yaml"))manifest = json.load(open("/kaggle/working/manifest.json"))train_data = manifest["train"]val_data   = manifest["val"]celeb_data = manifest["celeb_test"]BATCH_SIZE      = 16ACCUM_STEPS     = 4          # effective batch = 64EPOCHS          = 15         # reduced (early stopping will trigger earlier)LR              = 1e-5WEIGHT_DECAY    = 1e-2IMG_SIZE        = 256PATIENCE        = 5          # NEW: early stopping patienceLABEL_SMOOTHING = 0.05       # NEW: soft labelsWARMUP_EPOCHS   = 2          # NEW: freeze backbones during warmupprint(f"Train: {len(train_data):,} | Val: {len(val_data):,} | Celeb: {len(celeb_data):,}")print(f"Batch={BATCH_SIZE} x {ACCUM_STEPS} accum = {BATCH_SIZE*ACCUM_STEPS} effective")print(f"LR={LR} | WD={WEIGHT_DECAY} | LabelSmooth={LABEL_SMOOTHING}")print(f"EarlyStopping patience={PATIENCE} | Dropout=0.3 | DropPath=0.2")

In [ ]:
# == CELL 3: Fix Paths ==import globsample = train_data[0]recorded_path = sample.get("rgb_path", "N/A")print(f"Path in manifest : {recorded_path}")print(f"Exists now?      : {os.path.exists(recorded_path)}")found_npy = glob.glob("/kaggle/input/**/rgb/*.npy", recursive=True)if found_npy:    ACTUAL_PREP_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(found_npy[0])))    print(f"Detected root    : {ACTUAL_PREP_ROOT}")else:    ACTUAL_PREP_ROOT = None    print("No .npy files found!")def fix_path(p, new_root):    marker = "/preprocessed/"    return (new_root + p[p.index(marker):]) if marker in p else pdef remap_split(samples, new_root):    for s in samples:        for key in ["rgb_path", "fft_path", "hist_path"]:            if key in s: s[key] = fix_path(s[key], new_root)    return samplesif ACTUAL_PREP_ROOT and not os.path.exists(recorded_path):    train_data = remap_split(train_data, ACTUAL_PREP_ROOT)    val_data   = remap_split(val_data,   ACTUAL_PREP_ROOT)    celeb_data = remap_split(celeb_data, ACTUAL_PREP_ROOT)    print(f"Remapped -> {train_data[0]['rgb_path']}")valid_train = sum(1 for s in train_data if os.path.exists(s.get("rgb_path","")))assert valid_train > 0, "No valid training samples!"print(f"Valid train: {valid_train:,} / {len(train_data):,}")

In [ ]:
# == CELL 4: Dataset with STRONGER Augmentation ==MEAN = [0.485, 0.456, 0.406]STD  = [0.229, 0.224, 0.225]train_transform = transforms.Compose([    transforms.Resize((IMG_SIZE, IMG_SIZE)),    transforms.RandomHorizontalFlip(p=0.5),    transforms.RandomApply([        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05)    ], p=0.5),    transforms.RandomApply([        transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))    ], p=0.3),    transforms.RandomRotation(15),    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),    transforms.RandomGrayscale(p=0.05),    transforms.ToTensor(),    transforms.Normalize(MEAN, STD),    transforms.RandomErasing(p=0.15, scale=(0.02, 0.1)),])val_transform = transforms.Compose([    transforms.Resize((IMG_SIZE, IMG_SIZE)),    transforms.ToTensor(),    transforms.Normalize(MEAN, STD),])class NpyDataset(Dataset):    def __init__(self, samples, transform, name=""):        self.samples   = [s for s in samples if os.path.exists(s.get("rgb_path", ""))]        self.transform = transform        if len(self.samples) == 0:            raise RuntimeError("0 valid samples")        print(f"  {name}: {len(self.samples):,} samples")    def __len__(self): return len(self.samples)    def __getitem__(self, idx):        s = self.samples[idx]        label = torch.tensor(float(s["label"])).unsqueeze(0)        try:            rgb = np.load(s["rgb_path"])            return self.transform(Image.fromarray(rgb)), label        except Exception:            return torch.zeros(3, IMG_SIZE, IMG_SIZE), labelNW = config["data"]["num_workers"]train_ds   = NpyDataset(train_data, train_transform, "Train")val_ds     = NpyDataset(val_data,   val_transform,   "Val")celeb_ds   = NpyDataset(celeb_data, val_transform,   "Celeb")train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NW, pin_memory=True)val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NW, pin_memory=True)celeb_loader = DataLoader(celeb_ds, BATCH_SIZE, shuffle=False, num_workers=NW)print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

In [ ]:
# == CELL 5: Initialize Model ==import gcgc.collect(); torch.cuda.empty_cache()from model import EnsembleDeepfakeDetectorprint("Loading ConvNeXt-V2-Base + Swin-V2-Base (with Dropout)...")model = EnsembleDeepfakeDetector("/kaggle/working/model_config.yaml")if torch.cuda.device_count() > 1:    model = nn.DataParallel(model)model = model.to(device)total = sum(p.numel() for p in model.parameters())vram  = torch.cuda.memory_allocated() / 1e9print(f"Parameters    : {total:,}")print(f"VRAM (weights): {vram:.2f} GB")print("\nSmoke testing one batch...")model.train()x_test, y_test = next(iter(train_loader))x_test, y_test = x_test.to(device), y_test.to(device)with autocast():    m = model.module if hasattr(model, "module") else model    logits, _, _ = m(x_test)    _ = nn.BCEWithLogitsLoss()(logits, y_test)vram_peak = torch.cuda.max_memory_allocated() / 1e9print(f"Smoke test passed!  Peak VRAM: {vram_peak:.2f} GB")del x_test, y_test, logitstorch.cuda.empty_cache()

In [ ]:
# == CELL 6: Metric Helpers ==def sn34(y_true, y_prob):    y_pred     = (y_prob >= 0.5).astype(int)    norm_mcc   = (matthews_corrcoef(y_true, y_pred) + 1) / 2    norm_brier = 1 - brier_score_loss(y_true, y_prob)    return float(math.pow(max(norm_mcc,0)**1.2 * max(norm_brier,0)**1.8, 1/3.0))def evaluate(mdl, loader):    mdl.eval()    criterion  = nn.BCEWithLogitsLoss()    total_loss, probs, labels = 0.0, [], []    with torch.no_grad():        for x, y in loader:            x, y = x.to(device), y.to(device)            with autocast():                m = mdl.module if hasattr(mdl, "module") else mdl                logits, _, _ = m(x)                total_loss  += criterion(logits, y).item()            probs.extend(torch.sigmoid(logits.float()).cpu().numpy().flatten())            labels.extend(y.cpu().numpy().flatten())    y_true = np.array(labels)    y_prob = np.array(probs)    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else 0.5    return total_loss / len(loader), auc, sn34(y_true, y_prob), y_true, y_probprint("Metric helpers ready")

In [ ]:
# == CELL 7: Training Loop (Anti-Overfit) ==criterion = nn.BCEWithLogitsLoss()optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)# Proper warmup + cosine schedulewarmup_sched = optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)cosine_sched = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-WARMUP_EPOCHS, eta_min=1e-7)scheduler = optim.lr_scheduler.SequentialLR(optimizer, [warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])scaler = GradScaler()history, best_sn34_val, best_epoch = [], 0.0, 0no_improve = 0BEST = "/kaggle/working/ensemble_best.safetensors"def set_backbone_frozen(mdl, frozen=True):    m = mdl.module if hasattr(mdl, "module") else mdl    for param in m.cnn_branch.backbone.parameters():        param.requires_grad = not frozen    for param in m.vit_branch.backbone.parameters():        param.requires_grad = not frozen    trainable = sum(p.numel() for p in mdl.parameters() if p.requires_grad)    print(f"  Backbones {'FROZEN' if frozen else 'UNFROZEN'} | trainable: {trainable:,}")print(f"Training {EPOCHS} epochs | Early stopping patience={PATIENCE}")print("=" * 72)for epoch in range(1, EPOCHS + 1):    t0 = time.time()    if epoch == 1:        set_backbone_frozen(model, frozen=True)    elif epoch == WARMUP_EPOCHS + 1:        set_backbone_frozen(model, frozen=False)    model.train()    train_loss = 0.0    optimizer.zero_grad()    for step, (x, y) in enumerate(tqdm(train_loader, desc=f"Ep {epoch:02d}/{EPOCHS}", leave=False)):        x, y = x.to(device), y.to(device)        y_smooth = y * (1 - LABEL_SMOOTHING) + (1 - y) * LABEL_SMOOTHING        with autocast():            m = model.module if hasattr(model, "module") else model            logits, _, _ = m(x)            loss = criterion(logits, y_smooth) / ACCUM_STEPS        scaler.scale(loss).backward()        train_loss += loss.item() * ACCUM_STEPS        if (step + 1) % ACCUM_STEPS == 0:            scaler.unscale_(optimizer)            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)            scaler.step(optimizer)            scaler.update()            optimizer.zero_grad()    scheduler.step()    avg_train = train_loss / len(train_loader)    val_loss, val_auc, val_sn34, _, _ = evaluate(model, val_loader)    current_lr = optimizer.param_groups[0]["lr"]    gap = val_loss - avg_train    flag = ""    if val_sn34 > best_sn34_val:        best_sn34_val, best_epoch = val_sn34, epoch        (model.module if hasattr(model, "module") else model).save_safetensors(BEST)        no_improve = 0        flag = " BEST"    else:        no_improve += 1    rec = {"epoch": epoch, "train_loss": round(avg_train,5), "val_loss": round(val_loss,5),           "val_auc": round(val_auc,5), "val_sn34": round(val_sn34,5),           "lr": round(current_lr,8), "elapsed": round(time.time()-t0,1), "gap": round(gap,4)}    history.append(rec)    print(f"Ep {epoch:02d} | loss {avg_train:.4f}->{val_loss:.4f} (gap {gap:.3f}) | "          f"AUC {val_auc:.4f} | SN34 {val_sn34:.4f} | LR {current_lr:.2e}{flag}")    if no_improve >= PATIENCE:        print(f"\nEarly stopping at epoch {epoch} - no improvement for {PATIENCE} epochs")        breakjson.dump(history, open("/kaggle/working/training_history.json", "w"), indent=2)print(f"\nBest SN34 = {best_sn34_val:.4f}  (epoch {best_epoch})")

In [ ]:
# == CELL 8: Training Curves ==epochs_    = [h["epoch"]      for h in history]train_loss = [h["train_loss"] for h in history]val_loss_  = [h["val_loss"]   for h in history]val_auc_   = [h["val_auc"]    for h in history]val_sn34_  = [h["val_sn34"]   for h in history]lrs_       = [h["lr"]         for h in history]gaps_      = [h.get("gap",0)  for h in history]best_idx   = val_sn34_.index(max(val_sn34_))fig, axes = plt.subplots(1, 4, figsize=(22, 5))fig.patch.set_facecolor("#0d1117")def style(ax, title):    ax.set_facecolor("#161b22"); ax.tick_params(colors="white")    ax.set_title(title, color="white", fontsize=11)    ax.set_xlabel("Epoch", color="#8b949e")    for sp in ax.spines.values(): sp.set_color("#30363d")axes[0].plot(epochs_, train_loss, color="#58a6ff", lw=2, label="Train")axes[0].plot(epochs_, val_loss_,  color="#f85149", lw=2, label="Val")axes[0].legend(facecolor="#21262d", labelcolor="white")style(axes[0], "Loss (gap should stay small)")axes[1].plot(epochs_, val_auc_, color="#3fb950", lw=2)axes[1].axhline(0.95, color="#e3b341", ls="--", alpha=0.8, label="target 0.95")axes[1].set_ylim([0.7, 1.02])axes[1].legend(facecolor="#21262d", labelcolor="white")style(axes[1], "Validation AUC")axes[2].plot(epochs_, val_sn34_, color="#d2a8ff", lw=2)axes[2].fill_between(epochs_, val_sn34_, alpha=0.15, color="#d2a8ff")axes[2].scatter(epochs_[best_idx], val_sn34_[best_idx], color="#f0e68c", s=150, zorder=5,                label=f"Best {max(val_sn34_):.4f}")axes[2].legend(facecolor="#21262d", labelcolor="white")style(axes[2], "SN34")axes[3].plot(epochs_, gaps_, color="#ff7b72", lw=2)axes[3].axhline(0.2, color="#e3b341", ls="--", alpha=0.5, label="healthy gap")axes[3].legend(facecolor="#21262d", labelcolor="white")style(axes[3], "Train-Val Gap (overfit indicator)")fig.suptitle("Training v4 - Anti-Overfit", color="white", fontsize=13, fontweight="bold")plt.tight_layout()plt.savefig("/kaggle/working/training_curves.png", dpi=150, facecolor="#0d1117")plt.show()

In [ ]:
# == CELL 9: Confusion Matrix ==from model import EnsembleDeepfakeDetectorbest_model = EnsembleDeepfakeDetector("/kaggle/working/model_config.yaml").to(device)best_model.load_safetensors("/kaggle/working/ensemble_best.safetensors")_, _, _, y_true, y_prob = evaluate(best_model, val_loader)y_pred = (y_prob >= 0.5).astype(int)cm = confusion_matrix(y_true, y_pred)fig, axes = plt.subplots(1, 2, figsize=(14, 5))fig.patch.set_facecolor("#0d1117")axes[0].set_facecolor("#161b22")sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],            xticklabels=["Real","Fake"], yticklabels=["Real","Fake"],            linewidths=0.5, linecolor="#30363d", annot_kws={"size":16,"color":"white"})axes[0].set_title("Confusion Matrix (Val)", color="white", fontsize=13)axes[0].set_xlabel("Predicted", color="#8b949e")axes[0].set_ylabel("Actual", color="#8b949e")axes[0].tick_params(colors="white")metrics = {    "Precision": precision_score(y_true, y_pred, zero_division=0),    "Recall":    recall_score(y_true, y_pred, zero_division=0),    "F1":        f1_score(y_true, y_pred, zero_division=0),    "AUC":       roc_auc_score(y_true, y_prob),    "SN34":      sn34(y_true, y_prob),}axes[1].set_facecolor("#161b22")bars = axes[1].bar(metrics.keys(), metrics.values(),                   color=["#58a6ff","#3fb950","#d2a8ff","#f0e68c","#ff7b72"],                   edgecolor="#0d1117", width=0.5)axes[1].set_ylim([0, 1.15])axes[1].tick_params(colors="white")for sp in axes[1].spines.values(): sp.set_color("#30363d")for bar, v in zip(bars, metrics.values()):    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,                 f"{v:.3f}", ha="center", color="white", fontsize=11, fontweight="bold")axes[1].set_title("Model Metrics (Val)", color="white", fontsize=13)plt.suptitle("Model Evaluation", color="white", fontsize=14)plt.tight_layout()plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150, facecolor="#0d1117")plt.show()

In [ ]:
# == CELL 10: Cross-Dataset Benchmark ==_, celeb_auc, celeb_sn34, c_true, c_prob = evaluate(best_model, celeb_loader)c_pred = (c_prob >= 0.5).astype(int)print("Cross-Dataset: Celeb-DF v2 (unseen during training)")print(f"  AUC  : {celeb_auc:.4f}")print(f"  SN34 : {celeb_sn34:.4f}")print(f"  F1   : {f1_score(c_true, c_pred, zero_division=0):.4f}")summary = {    "architecture":    "ConvNeXtV2-Base + SwinV2-Base Ensemble (v4 Anti-Overfit)",    "best_epoch":       best_epoch,    "hyperparameters": {"lr": LR, "weight_decay": WEIGHT_DECAY,                        "batch_size": BATCH_SIZE, "img_size": IMG_SIZE,                        "label_smoothing": LABEL_SMOOTHING, "dropout": 0.3,                        "drop_path": 0.2, "patience": PATIENCE},    "in_distribution": {        "sn34": round(sn34(y_true, y_prob), 4),        "auc":  round(float(roc_auc_score(y_true, y_prob)), 4),        "f1":   round(float(f1_score(y_true, y_pred, zero_division=0)), 4),    },    "cross_dataset_celeb": {        "sn34": round(celeb_sn34, 4),        "auc":  round(celeb_auc, 4),        "f1":   round(float(f1_score(c_true, c_pred, zero_division=0)), 4),    },}json.dump(summary, open("/kaggle/working/model_summary.json", "w"), indent=2)print("\nmodel_summary.json saved")print("Proceed to Notebook 3 for explainability.")